# 이전 Q&A JSON → Quality Hub DB 이관

아래 코드 셀 맨 위의 `JSON_FILE`, `DB_HOST`, `DB_PORT`, `DB_USER`, `DB_NAME`만 실제 값으로 바꾼 뒤 셀을 한 번 실행하세요. DB 비밀번호는 실행할 때 별도 입력창으로 받습니다.

이 노트북과 `qna_json_migration.py`는 같은 `notes` 폴더에 두어야 합니다. 적재 중 한 건이라도 실패하면 전체가 롤백되며, 기존 `question_id`와 충돌하면 적재 전에 중단됩니다.

In [ ]:
# ================================================================
# 이 다섯 값만 실제 환경에 맞게 수정하세요.
# JSON_FILE은 원본 JSON 파일의 전체 경로를 권장합니다.
# ================================================================
JSON_FILE = r"/원본/파일/경로/legacy_qna.json"
DB_HOST = "DB서버주소"
DB_PORT = 3306
DB_USER = "DB사용자"
DB_NAME = "대상DB명"

# 아래 코드는 수정하지 않아도 됩니다.
import getpass
import importlib.util
import subprocess
import sys
from pathlib import Path

json_path = Path(JSON_FILE).expanduser().resolve()
if not json_path.is_file():
    raise FileNotFoundError(f"원본 JSON 파일을 찾을 수 없습니다: {json_path}")

# 노트북을 notes 폴더 또는 프로젝트 루트 어디에서 실행해도 변환 코드를 찾습니다.
module_candidates = [
    Path.cwd() / "qna_json_migration.py",
    Path.cwd() / "notes" / "qna_json_migration.py",
    json_path.parent / "qna_json_migration.py",
]
module_path = next((path for path in module_candidates if path.is_file()), None)
if module_path is None:
    raise FileNotFoundError(
        "qna_json_migration.py를 찾을 수 없습니다. 노트북과 같은 폴더에 두세요."
    )

spec = importlib.util.spec_from_file_location("qna_json_migration", module_path)
migration = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = migration
spec.loader.exec_module(migration)

report_path = json_path.with_name(f"{json_path.stem}.migration-report.json")
report = migration.MigrationReport(mode="notebook-apply", source_file=str(json_path))
connection = None

try:
    # 1. DB에 접속하기 전에 JSON 전체를 먼저 검증합니다.
    questions = migration.load_and_transform(json_path, report)
    if report.errors:
        raise migration.MigrationError(
            f"{len(report.errors)}개 행에 오류가 있습니다. 보고서의 errors를 확인하세요."
        )
    if not questions:
        raise migration.MigrationError("이관할 질문이 없습니다.")

    print("JSON 검증 완료")
    print(f"  질문: {report.valid_questions:,}건")
    print(f"  답변: {report.valid_messages:,}건")
    print(f"  경고: {len(report.warnings):,}건")
    if report.warning_counts:
        print(f"  경고 종류: {report.warning_counts}")

    # 2. 설치된 MySQL 드라이버가 없으면 현재 주피터 환경에 자동 설치합니다.
    try:
        import mysql.connector  # noqa: F401
    except ImportError:
        print("mysql-connector-python을 설치합니다...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "mysql-connector-python"
        ])

    db_password = getpass.getpass("DB 비밀번호: ")
    config = {
        "host": DB_HOST,
        "port": int(DB_PORT),
        "user": DB_USER,
        "password": db_password,
        "database": DB_NAME,
        "charset": "utf8mb4",
    }

    # 3. 스키마와 ID 충돌을 확인한 뒤 질문과 답변을 한 트랜잭션으로 적재합니다.
    connection, driver = migration.connect_database(config)
    report.target["driver"] = driver
    migration.apply_migration(connection, questions, report, allow_nonempty=True)
    report.status = "completed"

    print("\nDB 적재 완료")
    print(f"  질문: {report.inserted_questions:,}건")
    print(f"  답변: {report.inserted_messages:,}건")
    print(f"  대상 DB: {report.target.get('database')}")
    print(f"  결과 보고서: {report_path}")
except migration.MigrationError as error:
    report.status = "failed"
    report.failure = str(error)
    if connection is not None:
        connection.rollback()
    print(f"\n이관 중단: {error}")
    print(f"상세 보고서: {report_path}")
    raise
except Exception as error:
    report.status = "failed"
    report.failure = "database_or_runtime_error"
    if connection is not None:
        connection.rollback()
    error_code = getattr(error, "errno", None)
    if isinstance(error_code, int):
        report.target["error_code"] = error_code
    print("\n이관 실패: 전체 작업을 롤백했습니다.")
    print(f"상세 보고서: {report_path}")
    raise
finally:
    if connection is not None:
        connection.close()
    migration.write_report(report_path, report)
